In [ ]:
# Placeholder parameters: illustrative values only, not the ones used in live research.
WINDOW = 20  # 個人智慧財產,參數不提供 (placeholder)
Z_ENTRY_LONG = -2.0  # 個人智慧財產,參數不提供 (placeholder)
Z_ENTRY_SHORT = 2.0  # 個人智慧財產,參數不提供 (placeholder)
REV_LONG_Z = -2.0  # 個人智慧財產,參數不提供 (placeholder)
REV_SHORT_Z = 2.0  # 個人智慧財產,參數不提供 (placeholder)
VOL_REGIME_THRESHOLD = 30  # 個人智慧財產,參數不提供 (placeholder)
FLAT_Z = 0.25  # 個人智慧財產,參數不提供 (placeholder)
STRONG_SIZE = 1.0  # 個人智慧財產,參數不提供 (placeholder)
WEAK_SIZE = 0.25  # 個人智慧財產,參數不提供 (placeholder)

In [ ]:
!pip show vectorbt || pip install vectorbt

## Load Data from Google Drive

First, we need to mount Google Drive to access the historical data. After running the cell below, you will be prompted to authorize Colab to access your Google Drive files. Then, you can specify the path to your data files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
barrick_mining_path = 'data/Barrick Mining Stock Price History.csv'
gld_path = 'data/GLD ETF Stock Price History.csv'
gvz_path = 'data/CBOE Gold Volatitity Historical Data.csv' # Assuming this is the path to your GVZ data

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import vectorbt as vbt
import matplotlib.pyplot as plt

# ---------------------------
# LOAD CSVs
# ---------------------------
barrick = pd.read_csv(barrick_mining_path)
gld = pd.read_csv(gld_path)
gvz = pd.read_csv(gvz_path)

# ---------------------------
# CLEAN + SELECT ONLY PRICE
# ---------------------------
def clean_df(df, price_col_name):
    df["Date"] = pd.to_datetime(df["Date"])

    # Find correct price column (adjust if needed)
    if "Price" in df.columns:
        price_col = "Price"
    elif "Close" in df.columns:
        price_col = "Close"
    else:
        raise ValueError("No Price/Close column found")

    # Remove commas if present
    df[price_col] = df[price_col].astype(str).str.replace(',', '')
    df[price_col] = df[price_col].astype(float)

    df = df[["Date", price_col]].copy()
    df = df.rename(columns={price_col: price_col_name})

    df = df.set_index("Date").sort_index()
    return df

barrick = clean_df(barrick, "Barrick")
gld = clean_df(gld, "GLD")
gvz = clean_df(gvz, "GVZ")

# ---------------------------
# MERGE (now safe)
# ---------------------------
df = barrick.join(gld, how="inner").join(gvz, how="inner")
df = df.dropna().copy()

In [ ]:
# Logs
df["log_B"] = np.log(df["Barrick"])
df["log_G"] = np.log(df["GLD"])
df["log_GVZ"] = np.log(df["GVZ"])

# Interaction
df["interaction"] = df["log_G"] * df["log_GVZ"]

In [ ]:
X = df[["log_G", "log_GVZ", "interaction"]]
X = sm.add_constant(X)
y = df["log_B"]

model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
print(model.summary())

In [ ]:
beta1 = model.params["log_G"]
beta3 = model.params["interaction"]

df["beta_dynamic"] = beta1 + beta3 * df["log_GVZ"]

In [ ]:
df["ret_B"] = df["log_B"].diff()
df["ret_G"] = df["log_G"].diff()

df["spread_ret"] = df["ret_B"] - df["beta_dynamic"] * df["ret_G"]

df["spread"] = df["log_B"] - model.predict(X)

In [ ]:
window = WINDOW

df["z"] = (
    (df["spread"] - df["spread"].rolling(window).mean()) /
    df["spread"].rolling(window).std()
)

In [ ]:
df["signal"] = 0

# Mean reversion
df.loc[df["z"] > 2, "signal"] = -1
df.loc[df["z"] < -2, "signal"] = 1

# Exit
df.loc[df["z"].abs() < FLAT_Z, "signal"] = 0

# Regime (your key insight)
df["regime"] = 0
df.loc[df["GVZ"] > 25, "regime"] = 1

# Override in high vol
df.loc[df["regime"] == 1, "signal"] = 1

In [ ]:
df["position"] = df["signal"].shift(1).fillna(0)

df["strategy_ret"] = df["position"] * df["spread_ret"]
df["strategy_ret"] = df["strategy_ret"].fillna(0)

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df.index, df["z"])
plt.axhline(2, linestyle="--")
plt.axhline(-2, linestyle="--")
plt.title("Z-score")
plt.show()

### Z-score vs. Barrick Mining and GLD Prices

Let's visualize the calculated Z-score against the prices of Barrick Mining and GLD ETF. This will help in understanding how the spread's deviation from its mean (represented by the Z-score) correlates with the price movements of the individual assets.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Subplot X: Z-score vs. Barrick Mining Price
ax1 = axes[0]
sns.lineplot(x=df.index, y='Barrick', data=df, ax=ax1, color='blue', label='Barrick Mining Price')
ax1.set_ylabel('Barrick Mining Price', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_title('Z-score vs. Barrick Mining and GLD Prices')
ax1.grid(True)

ax1_twin = ax1.twinx() # Create a second Y-axis for Z-score
sns.lineplot(x=df.index, y='z', data=df, ax=ax1_twin, color='red', linestyle='--', label='Z-score')
ax1_twin.set_ylabel('Z-score', color='red')
ax1_twin.tick_params(axis='y', labelcolor='red')
ax1_twin.axhline(2, color='gray', linestyle=':', linewidth=0.8)
ax1_twin.axhline(-2, color='gray', linestyle=':', linewidth=0.8)
ax1_twin.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
ax1_twin.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
ax1_twin.legend(loc='upper left')


# Subplot X: Z-score vs. GLD ETF Price
ax2 = axes[1]
sns.lineplot(x=df.index, y='GLD', data=df, ax=ax2, color='green', label='GLD ETF Price')
ax2.set_xlabel('Date')
ax2.set_ylabel('GLD ETF Price', color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax2.grid(True)

ax2_twin = ax2.twinx() # Create a second Y-axis for Z-score
sns.lineplot(x=df.index, y='z', data=df, ax=ax2_twin, color='red', linestyle='--', label='Z-score')
ax2_twin.set_ylabel('Z-score', color='red')
ax2_twin.tick_params(axis='y', labelcolor='red')
ax2_twin.axhline(2, color='gray', linestyle=':', linewidth=0.8)
ax2_twin.axhline(-2, color='gray', linestyle=':', linewidth=0.8)
ax2_twin.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
ax2_twin.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
ax2_twin.legend(loc='upper left')

plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()

In [ ]:
import numpy as np
import vectorbt as vbt

# ---------------------------
# PARAMETERS
# ---------------------------
z_entry_long = Z_ENTRY_LONG
z_entry_short = Z_ENTRY_SHORT
gvz_threshold = VOL_REGIME_THRESHOLD

# ---------------------------
# SIGNAL COMPONENTS
# ---------------------------
z = df["z"]
z_diff = z.diff()

# Reversal-based entry (fix timing)
long_entry = (z < REV_LONG_Z) & (z_diff > 0)
short_entry = (z > REV_SHORT_Z) & (z_diff < 0)
# Exit rules (mean reversion complete)
long_exit = (z > 0)
short_exit = (z < 0)

# Regime filter (disable trading)
regime_ok = df["GVZ"] < gvz_threshold

# ---------------------------
# POSITION (EVENT-DRIVEN)
# ---------------------------
position = np.zeros(len(df))

for i in range(1, len(df)):

    # If no position
    if position[i-1] == 0:
        if long_entry.iloc[i] and regime_ok.iloc[i]:
            position[i] = STRONG_SIZE     # strong long
        elif short_entry.iloc[i] and regime_ok.iloc[i]:
            position[i] = -WEAK_SIZE    # weaker short

    # If long
    elif position[i-1] > 0:
        if long_exit.iloc[i] or not regime_ok.iloc[i]:
            position[i] = 0
        else:
            position[i] = position[i-1]

    # If short
    elif position[i-1] < 0:
        if short_exit.iloc[i] or not regime_ok.iloc[i]:
            position[i] = 0
        else:
            position[i] = position[i-1]

df["position"] = position

# ---------------------------
# STRATEGY RETURNS (CORRECT)
# ---------------------------
df["strategy_ret"] = df["position"].shift(1) * df["spread_ret"]
df["strategy_ret"] = df["strategy_ret"].fillna(0)

# ---------------------------
# VECTORBT BACKTEST (CORRECT USAGE)
# ---------------------------
price = df["Barrick"]

# Entries / exits (clean event detection)
entries = (df["position"].shift(1) == 0) & (df["position"] > 0)
exits = (df["position"].shift(1) > 0) & (df["position"] == 0)

short_entries = (df["position"].shift(1) == 0) & (df["position"] < 0)
short_exits = (df["position"].shift(1) < 0) & (df["position"] == 0)

# Position sizing (important)
size = df["position"].abs()

pf = vbt.Portfolio.from_signals(
    close=price,
    entries=entries,
    exits=exits,
    short_entries=short_entries,
    short_exits=short_exits,
    size=size,              # <-- THIS matters
    freq="1D"
)

# OUTPUT
print(pf.stats())
pf.plot().show()

# ---------------------------
# OUTPUT
# ---------------------------
print(pf.stats())
pf.plot().show()